In [ ]:
#!/usr/bin/env python3
"""
Weiji's script
Batch process common video formats to generate subtitles using an AI model 'Whisper'.

This script will:
1. Scan a directory for all video files supported by ffmpeg (e.g., .flv, .mp4, .mkv, etc.).
2. Extract audio from each video using ffmpeg.
3. Transcribe the audio and generate an SRT subtitle file via Whisper.
4. Save the transcription text and subtitles into an output folder.
5. Burn subtitles into the original video, producing new videos in the same format in the output directory.
6. Clean up intermediate .wav and .srt files, keeping only the original videos, subtitled videos, and .txt transcriptions.

Setup and installation (run these commands in your terminal):

# 1. Activate your conda environment. for example
conda activate jimnew

# 2. Install ffmpeg for audio extraction
conda install -c conda-forge ffmpeg

# 3. Install Git if needed (for whisper cloning)
conda install -c conda-forge git

# 4. Clone Whisper repository and install
git clone https://github.com/openai/whisper.git
cd whisper
pip install -U pip setuptools wheel
pip install -r requirements.txt
pip install -e .

# 5. Verify installation
whisper --help
"""

import os

# --- 必须放在所有 import 之前 ---
# 屏蔽 0 号 Blackwell 显卡，只使用 1 号和 2 号 A4500
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import glob
import subprocess

import torch
import whisper



# 再次验证，此时的 '显卡 0' 实际上会是系统里的 A4500
if torch.cuda.is_available():
    print(f"成功屏蔽不兼容显卡。当前可用显卡数量: {torch.cuda.device_count()}")
    print(f"当前主显卡: {torch.cuda.get_device_name(0)}")
# Convert seconds to SRT timestamp format: HH:MM:SS,mmm
def format_timestamp(seconds: float) -> str:
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hrs:02}:{mins:02}:{secs:02},{millis:03}"


# Write Whisper transcription segments into an SRT file
def write_srt(segments: list, srt_path: str):
    with open(srt_path, "w", encoding="utf-8") as f:
        for index, seg in enumerate(segments, start=1):
            start_ts = format_timestamp(seg["start"])
            end_ts = format_timestamp(seg["end"])
            text = seg["text"].strip()
            f.write(f"{index}\n{start_ts} --> {end_ts}\n{text}\n\n")


def get_device() -> str:
    """
    Decide which device to use.

    - Prefer CUDA if available and usable.
    - If CUDA is unavailable or misconfigured (like invalid device / no kernel image),
      fall back to CPU.
    """
    try:
        if not torch.cuda.is_available():
            print("[Device] CUDA not available, using CPU.")
            return "cpu"

        # Try a minimal CUDA operation to detect misconfiguration early
        try:
            _ = torch.randn(1, device="cuda:0")
        except Exception as e:
            print(f"[Device] CUDA appears misconfigured ({e}), falling back to CPU.")
            return "cpu"

        print("[Device] Using CUDA (GPU).")
        return "cuda"
    except Exception as e:
        print(f"[Device] Error while checking CUDA: {e}")
        print("[Device] Falling back to CPU.")
        return "cpu"


# Base directory containing video files
base_dir = "/home/weiji/whisper/Weijidata"
# Directory to store output files
output_dir = os.path.join(base_dir, "output")
# Create output directory if it does not exist
os.makedirs(output_dir, exist_ok=True)

# Decide device and load the Whisper model (choose "small", "medium", or "large")
device = get_device()
model_name = "large"

print(f"[Whisper] Loading model '{model_name}' on device: {device}")
try:
    model = whisper.load_model(model_name, device=device)
except Exception as e:
    # If loading on CUDA fails (e.g., no kernel image available), retry on CPU
    if device != "cpu":
        print(f"[Whisper] Failed to load model on {device}: {e}")
        print("[Whisper] Retrying on CPU...")
        device = "cpu"
        model = whisper.load_model(model_name, device=device)
    else:
        raise

print(f"[Whisper] Model loaded on device: {device}")

# Find all video files supported by ffmpeg in the base directory
video_extensions = (
    '.flv', '.mp4', '.mkv', '.mov', '.avi', '.webm',
    '.mpg', '.mpeg', '.wmv', '.m4v', '.m4a'
)

video_files = []
for ext in video_extensions:
    video_files.extend(glob.glob(os.path.join(base_dir, f"*{ext}")))

if not video_files:
    print(f"No video/audio files found in {base_dir} with extensions: {video_extensions}")
else:
    print(f"Found {len(video_files)} files to process.")

for video_path in video_files:
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    ext = os.path.splitext(video_path)[1]  # keep original extension
    wav_path = os.path.join(base_dir, f"{base_name}.wav")
    txt_path = os.path.join(output_dir, f"{base_name}.txt")
    srt_path = os.path.join(output_dir, f"{base_name}.srt")
    subtitled_video = os.path.join(output_dir, f"{base_name}_subtitled{ext}")  # output in same format

    print(f"\nProcessing: {base_name}{ext}")

    # Extract audio from video using ffmpeg (16 kHz, mono)
    print(f"  Extracting audio to: {wav_path}")
    subprocess.run([
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", wav_path
    ], check=True)

    # Transcribe audio with Whisper
    print("  Transcribing audio with Whisper...")
    use_fp16 = (device == "cuda")
    result = model.transcribe(
        wav_path,
        language="English",
        fp16=use_fp16
    )

    # Save transcription text
    print(f"  Saving transcription text to: {txt_path}")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(result["text"])

    # Save SRT subtitle file
    print(f"  Saving subtitles to: {srt_path}")
    write_srt(result["segments"], srt_path)

    # Burn subtitles into the original video using ffmpeg
    print(f"  Burning subtitles into video: {subtitled_video}")
    audio_only_exts = {".m4a", ".mp3", ".wav"}

    if ext.lower() in audio_only_exts:
        print("  Audio-only file detected — skipping subtitle burning.")
    else:
        subprocess.run([
            "ffmpeg", "-y", "-i", video_path,
            "-vf", f"subtitles={srt_path}",
            "-c:a", "copy", subtitled_video
        ], check=True)

    # Clean up intermediate files
    print(f"  Cleaning up intermediate files: {wav_path}, {srt_path}")
    try:
        os.remove(wav_path)
    except FileNotFoundError:
        print(f"  Warning: {wav_path} not found during cleanup.")
    try:
        os.remove(srt_path)
    except FileNotFoundError:
        print(f"  Warning: {srt_path} not found during cleanup.")

    print(f"Completed: subtitled video and transcription saved to {output_dir}")

print("\nAll videos have been processed.")


In [ ]:
import os
import torch

# 强制只让程序看到两张 A4500 显卡，跳过那张不兼容的 PRO 6000
# 根据你之前 nvidia-smi 的顺序，A4500 通常是 0 和 1
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# 验证一下
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"[Check] 正在使用显卡 {i}: {torch.cuda.get_device_name(i)}")